# Fleet Maintenance Planning — V6


Predict vehicle failures and repair costs, then optimize inspection decisions and budgets. Run all cells in order. The final submission is `sub_v6.csv`; earlier submission variants are retained for reproducibility.

All supervised models are fitted on labelled training data. 

In [ ]:
import os, glob, time, warnings
import numpy as np, pandas as pd
import lightgbm as lgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.linear_model import LogisticRegression, LinearRegression, BayesianRidge
warnings.filterwarnings("ignore")
SEED = 2024; T0 = time.time()
INSPECT, RUSH, RETURN = 10.0, 0.20, 0.02
Q_STAR = RUSH / (RUSH + RETURN)
cands = sorted(glob.glob("/kaggle/input/**/train.csv", recursive=True)) + \
        [p for p in ["data/train.csv", "../data/train.csv", "../train.csv", "train.csv"] if os.path.exists(p)]
DATA_DIR = os.path.dirname(cands[0]) or "."
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
tr = pd.read_csv(f"{DATA_DIR}/train.csv", dtype={"id": str}); te = pd.read_csv(f"{DATA_DIR}/test.csv", dtype={"id": str})
y, fu, cost = tr.failure.values, tr.followup_days.values, tr.repair_cost.values
FULL = fu == 365; LOGC = np.log(np.nan_to_num(cost, nan=1.0))
SENSORS = [f"s{i:03d}" for i in range(170)]; HIST = [f"s{i:03d}" for i in range(54, 64)]
print(tr.shape, te.shape, "| failures", y.sum())

In [ ]:
def align(df):                      
    df = df.copy(); old = df[HIST].copy()
    for k in range(54, 63): df[f"s{k:03d}"] = old[f"s{k+1:03d}"].values
    df["s063"] = np.where(old["s054"].isna(), np.nan, 0.0); return df
teA = align(te); trH = tr.copy()
for c, m in {"s055": 10, "s057": 100, "s062": 10}.items(): trH[c] = np.round(trH[c] / m) * m
F_ROB = [c for c in SENSORS if c not in ("s047", "s120", "s122")]
STABLE = [c for c in F_ROB if c not in HIST and c != "s146"]

## Sensor distribution analysis


In [ ]:
cols = ["s123", "s012", "s097", "s164"]
display(pd.concat({"train, followup=365": tr[FULL][cols].median(), "train, followup<365": tr[~FULL][cols].median(),
                   "train, all": tr[cols].median(), "test": te[cols].median()}, axis=1).round(0))

## Failure classification and cross-validation

In [ ]:
def stable_features(df):
    X = df[STABLE].copy()
    X["n_missing"] = df[STABLE].isna().sum(axis=1); X["n_zero"] = (df[STABLE] == 0).sum(axis=1)
    X["log_row_sum"] = np.log1p(df[STABLE].fillna(0).sum(axis=1)); return X
XS, XS_test = stable_features(tr), stable_features(te)
W180 = np.where((y == 0) & (fu < 180), 0.0, 1.0)
def make_folds(seed, repeats):
    return [list(StratifiedKFold(5, shuffle=True, random_state=seed + r).split(y, y)) for r in range(repeats)]
FOLDS = make_folds(SEED, 3)
def ensemble(s):
    return [lgb.LGBMClassifier(n_estimators=700, learning_rate=0.02, num_leaves=15, min_child_samples=30, subsample=0.8,
                subsample_freq=1, colsample_bytree=0.4, reg_lambda=5.0, verbose=-1, n_jobs=2, random_state=s),
            XGBClassifier(n_estimators=700, learning_rate=0.02, max_depth=5, min_child_weight=3, subsample=0.8,
                colsample_bytree=0.4, reg_lambda=5.0, tree_method="hist", eval_metric="logloss", n_jobs=2, verbosity=0, random_state=s),
            CatBoostClassifier(iterations=800, learning_rate=0.04, depth=6, l2_leaf_reg=5.0, rsm=0.5, verbose=0,
                thread_count=2, allow_writing_files=False, random_seed=s)]
P, PT = np.zeros((3, len(y))), np.zeros(len(te))
for r, fs in enumerate(FOLDS):
    for k, (a, b) in enumerate(fs):
        for m in ensemble(SEED + 10 * r + k):
            m.fit(XS.iloc[a], y[a], sample_weight=W180[a])
            P[r, b] += m.predict_proba(XS.iloc[b])[:, 1] / 3; PT += m.predict_proba(XS_test)[:, 1] / 45
    print("repeat", r, f"{time.time()-T0:.0f}s")

class Platt:
    @staticmethod
    def z(p):
        p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))[:, None]
    def fit(self, p, yy): self.m = LogisticRegression(C=1e4, max_iter=5000).fit(self.z(p), yy); return self
    def predict(self, p): return self.m.predict_proba(self.z(p))[:, 1]
def cross_cal(p, fs):
    out = np.zeros_like(p)
    for a, b in fs:
        a = a[FULL[a]]; out[b] = Platt().fit(p[a], y[a]).predict(p[b])
    return out
Pc = [cross_cal(P[r], FOLDS[r]) for r in range(3)]
final_cal = Platt().fit(P.mean(0)[FULL], y[FULL])
pt_cal, ptr_cal = final_cal.predict(PT), final_cal.predict(P.mean(0))

In [ ]:
pfull = np.zeros(len(y))
for a, b in StratifiedKFold(5, shuffle=True, random_state=1).split(XS, FULL):
    m = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.03, num_leaves=15, verbose=-1, n_jobs=1).fit(XS.iloc[a], FULL[a])
    pfull[b] = m.predict_proba(XS.iloc[b])[:, 1]
WIP = 1 / np.clip(pfull, 0.2, 1)
PI_FULL = y[FULL].mean()
PI_IPCW = (WIP * y)[FULL].sum() / WIP[FULL].sum()
PI_REF = ptr_cal.mean()
def adjust_prior(p, a, b):
    r1, r0 = b / a, (1 - b) / (1 - a); return r1 * p / (r1 * p + r0 * (1 - p))
def em_prior(p, a):
    pi = a
    for _ in range(1000):
        new = adjust_prior(p, a, pi).mean()
        if abs(new - pi) < 1e-9: break
        pi = new
    return pi
PI_TEST = em_prior(pt_cal, PI_REF)
ODDS = (PI_TEST / (1 - PI_TEST)) / (PI_REF / (1 - PI_REF))
print(f"train rate: full-follow-up {PI_FULL:.4f} | all vehicles: IPCW {PI_IPCW:.4f}, mean calibrated p {PI_REF:.4f}")
print(f"test rate (EM) {PI_TEST:.4f}  ->  odds multiplier {ODDS:.2f}  (V3b used about 5.4)")

def mixture(ref, q):
    o = []
    for c in STABLE:
        t = ref[c].quantile(q); a = (ref[c] > t).mean(); b = (tr[y == 1][c] > t).mean(); f = (te[c] > t).mean()
        if b - a > 0.2: o.append((f - a) / (b - a))
    return np.median(o)
rows = []
for q in [0.90, 0.95, 0.98, 0.99, 0.995]:
    extra = mixture(tr, q)
    rows.append((q, mixture(tr[FULL & (y == 0)], q), extra + (1 - extra) * PI_IPCW))
display(pd.DataFrame(rows, columns=["threshold quantile", "ref: full-follow-up healthy", "ref: all vehicles"]).round(4))

## Baseline repair-cost model

Fit log repair costs on training failures using log-transformed sensors, histogram features, missing counts and 60 log-ratios selected within each fitting fold. Median imputation is computed across the training dataset before cross-validation.

In [ ]:
SR = F_ROB; med = trH[SR].median()
L, LT = np.log1p(trH[SR].fillna(med)).values, np.log1p(teA[SR].fillna(med)).values
def hist_feats(df):
    H = df[HIST].fillna(0).values; t = H.sum(1); Q = H / np.maximum(t, 1)[:, None]
    return np.column_stack([Q, np.cumsum(Q, 1)[:, 1:8], np.log1p(t), (Q * np.arange(10)).sum(1)])
HF, HFT = hist_feats(trH), hist_feats(teA)
NA, NAT = trH[SR].isna().sum(1).values[:, None], teA[SR].isna().sum(1).values[:, None]

def top_ratios(Lt, yt, k):
    Lc = Lt - Lt.mean(0); yc = yt - yt.mean(); cov = Lc.T @ yc; G = Lc.T @ Lc; d = np.diag(G)
    var = d[:, None] + d[None, :] - 2 * G; num = cov[:, None] - cov[None, :]
    C = np.where(var > 1e-8, np.abs(num) / np.sqrt(np.maximum(var, 1e-8) * (yc ** 2).sum()), 0); C = np.triu(C, 1)
    use, out = np.zeros(Lt.shape[1], int), []
    for f in np.argsort(-C, axis=None):
        i, j = divmod(int(f), Lt.shape[1])
        if use[i] < 3 and use[j] < 3: out.append((i, j)); use[i] += 1; use[j] += 1
        if len(out) >= k: break
    return out

def scaler(A):
    mu, sd = A.mean(0), A.std(0) + 1e-9
    return lambda B: np.clip((B - mu) / sd, -5, 5)

# every fit_* returns predict(indices, test=False)
def fit_bayes_top(idx, yy, seed):          # Bayesian ridge on the 80 inputs most correlated with cost
    def build(j, t): return np.column_stack([(LT if t else L)[j], (HFT if t else HF)[j]])
    A = build(idx, False); Ac = A - A.mean(0); yc = yy - yy.mean()
    c = np.abs(Ac.T @ yc) / np.sqrt((Ac ** 2).sum(0) * (yc ** 2).sum() + 1e-30); sel = np.argsort(-c)[:80]
    nz = scaler(A[:, sel]); m = BayesianRidge().fit(nz(A[:, sel]), yy)
    return lambda j, t=False: m.predict(nz(build(j, t)[:, sel]))
def fit_bayes_all(idx, yy, seed):
    def build(j, t): return np.column_stack([(LT if t else L)[j], (HFT if t else HF)[j]])
    A = build(idx, False); nz = scaler(A); m = BayesianRidge().fit(nz(A), yy)
    return lambda j, t=False: m.predict(nz(build(j, t)))
def fit_lgb_ratio(idx, yy, seed):
    pr = top_ratios(L[idx], yy, 60)
    def build(j, t):
        M = (LT if t else L)[j]
        return np.column_stack([M, (HFT if t else HF)[j], (NAT if t else NA)[j],
                                np.column_stack([M[:, a] - M[:, b] for a, b in pr])])
    m = lgb.LGBMRegressor(n_estimators=700, learning_rate=0.02, num_leaves=7, min_child_samples=5, subsample_freq=1,
                          colsample_bytree=0.25, reg_lambda=2.0, verbose=-1, n_jobs=1, random_state=seed).fit(build(idx, False), yy)
    return lambda j, t=False: m.predict(build(j, t))
def fit_boosted_residual(idx, yy, seed):
    inner = np.zeros(len(idx))
    for a, b in KFold(5, shuffle=True, random_state=seed).split(idx):
        inner[b] = fit_bayes_all(idx[a], yy[a], seed)(idx[b])
    f0, f1 = fit_bayes_all(idx, yy, seed), fit_lgb_ratio(idx, yy - inner, seed)
    return lambda j, t=False: f0(j, t) + f1(j, t)
def fit_lgb_v3(idx, yy, seed):             # V3 cost model, for comparison
    A, AT = (pd.concat([d[F_ROB], d[F_ROB].isna().sum(axis=1).rename("n_nan"), (d[F_ROB] == 0).sum(axis=1).rename("n_zero")], axis=1)
             for d in (trH, teA))
    m = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.03, num_leaves=7, min_child_samples=8, subsample_freq=1,
                          colsample_bytree=0.5, verbose=-1, n_jobs=1, random_state=seed).fit(A.iloc[idx], yy)
    return lambda j, t=False: m.predict((AT if t else A).iloc[j])
V4_COST = [fit_bayes_top, fit_lgb_ratio, fit_boosted_residual]

def fit_cost(members, idx, seed):
    # blend + linear recalibration; residual pool from an inner 5-fold CV
    yy = LOGC[idx]; inner = np.zeros(len(idx))
    for a, b in KFold(5, shuffle=True, random_state=seed).split(idx):
        for f in members: inner[b] += f(idx[a], yy[a], seed)(idx[b]) / len(members)
    lin = LinearRegression().fit(inner[:, None], yy)
    fitted = [f(idx, yy, seed) for f in members]
    pred = lambda j, t=False: lin.predict((sum(g(j, t) for g in fitted) / len(fitted))[:, None])
    return pred, yy - lin.predict(inner[:, None])

##  Inspection and budget decision rule

Use the 0.20/0.22 quantile of the predictive cost mixture, including the probability mass at zero. Inspect when expected inspection cost is lower than expected uninspected repair cost. V4 cost samples are clipped to [5, 8000].

In [ ]:
def fleet_cost(flag, budget, failure, repair_cost):
    c = np.where(failure == 1, np.nan_to_num(repair_cost), 0.0)
    return np.where(flag == 1, INSPECT + RUSH * np.maximum(c - budget, 0) + RETURN * np.maximum(budget - c, 0), c)
def decide(p, mu, resid, clip=True):
    r = np.sort(resid); R = len(r); F, B = np.zeros(len(p), int), np.zeros(len(p))
    for s in range(0, len(p), 4000):
        e = slice(s, s + 4000); pe = p[e]; C = np.exp(mu[e][:, None] + r[None, :])
        if clip: C = np.clip(C, 5, 8000)
        level = np.where(pe > 1 - Q_STAR, (pe - (1 - Q_STAR)) / np.maximum(pe, 1e-12), 0.0)
        idx = np.clip(np.ceil(level * R).astype(int) - 1, 0, R - 1)
        b = np.where(level > 0, C[np.arange(len(pe)), idx], 0.0)
        ec = INSPECT + pe * (RUSH * np.maximum(C - b[:, None], 0).mean(1) + RETURN * np.maximum(b[:, None] - C, 0).mean(1)) + (1 - pe) * RETURN * b
        f = (ec < pe * C.mean(1)).astype(int); F[e] = f; B[e] = np.where(f == 1, np.round(b, 2), 0.0)
    return F, B

## Simulated-depot validation

Compare cost models on six alternative splits using inverse censoring weights and assumed failure rates and repair-cost levels. The V3-style reference is a reimplementation. The same 397 failures are reused across model development and evaluation; selection optimism may remain. These comparisons are not independent end-to-end holdout estimates.

In [ ]:
def weights(pi):
    W = np.where(FULL, WIP * np.where(y == 1, pi / PI_IPCW, (1 - pi) / (1 - PI_IPCW)), 0.0); return W / W.sum()
WORLDS = [(0.025, 1.0), (0.033, 1.0), (0.040, 1.0), (0.050, 1.0), (0.033, 0.8), (0.025, 0.6), (0.040, 1.3)]
CONFIGS = {"V3-style": ([fit_lgb_v3], 5.42, False),
           "V3-style, new prior": ([fit_lgb_v3], ODDS, False),
           "V4a": (V4_COST, ODDS, True), "V4b (odds 3.0)": (V4_COST, 3.0, True)}
fresh = make_folds(7777, 6); res = {k: np.zeros((6, len(WORLDS))) for k in CONFIGS}
for r, fs in enumerate(fresh):
    preds = {}
    for name, members in [("v3", [fit_lgb_v3]), ("v4", V4_COST)]:
        preds[name] = [fit_cost(members, a[y[a] == 1], SEED + 10 * r + k) for k, (a, b) in enumerate(fs)]
    for cfg, (members, odds, clip) in CONFIGS.items():
        F, B = np.zeros(len(y), int), np.zeros(len(y))
        for k, (a, b) in enumerate(fs):
            m = b[FULL[b]]; pred, rs = preds["v4" if members is V4_COST else "v3"][k]
            o = odds * Pc[r % 3][m] / (1 - Pc[r % 3][m])
            F[m], B[m] = decide(o / (1 + o), pred(m), rs, clip)
        for j, (pi, cs) in enumerate(WORLDS):
            res[cfg][r, j] = 20000 * (weights(pi) * fleet_cost(F, B, y, cost * cs)).sum()
    print("fresh split", r, f"{time.time()-T0:.0f}s")
idx = pd.MultiIndex.from_tuples(WORLDS, names=["failure rate", "cost level"])
print("Mean cost per 20,000 vehicles"); display(pd.DataFrame({k: v.mean(0) for k, v in res.items()}, index=idx).round(0))
print("Paired difference vs V3-style (negative = better)")
display(pd.DataFrame({k: (v - res["V3-style"]).mean(0) for k, v in res.items()}, index=idx).round(0))
print("Splits (out of 6) on which it beats V3-style")
display(pd.DataFrame({k: (v < res["V3-style"]).sum(0) for k, v in res.items()}, index=idx))

In [ ]:
fails = np.where(y == 1)[0]; mus, pools = [], []
for s in range(5):
    pred, rs = fit_cost(V4_COST, fails, SEED + s)
    mus.append(pred(np.arange(len(te)), True)); pools.append(rs)
mu_test, pool = np.mean(mus, 0), np.mean(pools, 0)
for name, odds in [("v4a", ODDS), ("v4b", 3.0)]:
    o = odds * pt_cal / (1 - pt_cal); p = o / (1 + o)
    F, B = decide(p, mu_test, pool)
    sub = pd.DataFrame({"id": te.id, "flag": F, "budget": B})
    assert len(sub) == len(te) and (sub.id.values == te.id.values).all() and sub.flag.isin([0, 1]).all()
    assert np.isfinite(sub.budget).all() and (sub.budget >= 0).all()
    sub.to_csv(f"{OUT_DIR}/sub_{name}.csv", index=False)
    print(f"sub_{name}.csv: flagged {F.sum()}, expected failures {p.sum():.0f}, median budget of flagged {np.median(B[F==1]):.0f}, max {B.max():.0f}")
print(f"total {time.time()-T0:.0f}s")

In [ ]:
base = pd.read_csv(f"{OUT_DIR}/sub_v4a.csv", dtype={"id": str})
for name, scale in [("v5", 1.15), ("v5_high", 1.30)]:
    sub = base.copy(); sub["budget"] = np.minimum(sub["budget"] * scale, 8000).round(2)
    assert (sub.flag.values == base.flag.values).all() and (sub.budget >= 0).all() and len(sub) == len(te)
    sub.to_csv(f"{OUT_DIR}/sub_{name}.csv", index=False)
    print(f"sub_{name}.csv: flagged {sub.flag.sum()}, budget sum {sub.budget.sum():,.0f}, max {sub.budget.max():,.0f}")

##  Final V6 cost model

Use seven histogram-like sensor blocks (s023–s032, s037–s046, s054–s063, s080–s089, s109–s118, s129–s138 and s154–s163). Joint missingness is checked on both unlabelled test and training features. Extract shares, cumulative shares, log totals, mean bins, spread, entropy and maximum share.

Average RBF kernel ridge on 60 selected features, Bayesian ridge with LightGBM residual correction, and LightGBM with 100 selected log-ratios. Recalibrate predictions and estimate per-vehicle residual spread from ensemble disagreement. Apply the V5 budget multiplier of 1.15.

In [ ]:
STARTS = [23, 37, 54, 80, 109, 129, 154]
BLOCKS = [[f"s{i:03d}" for i in range(s, s + 10)] for s in STARTS]
both = pd.concat([tr[SENSORS], te[SENSORS]])
for bl in BLOCKS:
    miss = both[bl].isna().values
    assert (miss.all(1) == miss.any(1)).all(), bl          # a block is missing entirely or not at all
def block_feats(df):
    out, k = [], np.arange(10)
    for bl in BLOCKS:
        H = df[bl].values.astype(float); miss = np.isnan(H).all(1); H = np.nan_to_num(H); t = H.sum(1); Q = H / np.maximum(t, 1)[:, None]
        cum = np.cumsum(Q, 1); mean = (Q * k).sum(1); sd = np.sqrt((Q * (k - mean[:, None]) ** 2).sum(1)); ent = -(Q * np.log(Q + 1e-9)).sum(1)
        F = np.column_stack([Q, cum[:, 1:8], np.log1p(t), mean, sd, ent, Q.max(1)]); F[miss] = 0; out.append(F)
    tot = np.column_stack([o[:, 17] for o in out]); pos = np.column_stack([o[:, 18] for o in out])
    return np.column_stack(out + [tot - np.median(tot, 1, keepdims=True), pos.mean(1), tot.mean(1)])
X7, X7T = block_feats(trH), block_feats(teA)
print("block features:", X7.shape[1])

In [ ]:
from sklearn.kernel_ridge import KernelRidge
def top_corr(A, yy, k):
    Ac = A - A.mean(0); yc = yy - yy.mean()
    return np.argsort(-np.abs(Ac.T @ yc) / np.sqrt((Ac ** 2).sum(0) * (yc ** 2).sum() + 1e-30))[:k]

def fit_kernel(idx, yy, seed):
    A = X7[idx]; sel = top_corr(A, yy, 60); nz = scaler(A[:, sel]); off = yy.mean()
    m = KernelRidge(alpha=0.1, kernel="rbf", gamma=0.15 / 60).fit(nz(A[:, sel]), yy - off)
    return lambda j, t=False: m.predict(nz((X7T if t else X7)[j][:, sel])) + off

def fit_bayes_blocks(idx, yy, seed):
    def build(j, t): return np.column_stack([(LT if t else L)[j], (X7T if t else X7)[j]])
    A = build(idx, False); nz = scaler(A); off = yy.mean(); m = BayesianRidge().fit(nz(A), yy - off)
    return lambda j, t=False: m.predict(nz(build(j, t))) + off

def lgb_blocks(idx, yy, seed, use_logs, trees, colsample):
    pr = top_ratios(L[idx], yy, 100)
    def build(j, t):
        M = (LT if t else L)[j]
        cols = ([M] if use_logs else []) + [(X7T if t else X7)[j], (NAT if t else NA)[j],
                np.column_stack([M[:, a] - M[:, b] for a, b in pr])]
        return np.column_stack(cols)
    m = lgb.LGBMRegressor(n_estimators=trees, learning_rate=0.02, num_leaves=7, min_child_samples=5, subsample_freq=1,
                          colsample_bytree=colsample, reg_lambda=2.0, verbose=-1, n_jobs=1, random_state=seed).fit(build(idx, False), yy)
    return lambda j, t=False: m.predict(build(j, t))

def fit_residual_blocks(idx, yy, seed):
    inner = np.zeros(len(idx))
    for a, b in KFold(5, shuffle=True, random_state=seed).split(idx):
        inner[b] = fit_bayes_blocks(idx[a], yy[a], seed)(idx[b])
    f0 = fit_bayes_blocks(idx, yy, seed); f1 = lgb_blocks(idx, yy - inner, seed, False, 700, 0.25)
    return lambda j, t=False: f0(j, t) + f1(j, t)

def fit_lgb_wide(idx, yy, seed): return lgb_blocks(idx, yy, seed, True, 900, 0.12)

V6_COST = [fit_kernel, fit_residual_blocks, fit_lgb_wide]

def fit_cost_spread(idx, seed):
    """returns predict(indices, test) -> (mu, sigma) and the standardised residual pool z"""
    yy = LOGC[idx]; inner = np.zeros((3, len(idx)))
    for a, b in KFold(5, shuffle=True, random_state=seed).split(idx):
        for n, f in enumerate(V6_COST): inner[n, b] = f(idx[a], yy[a], seed)(idx[b])
    mean_in = inner.mean(0); lin = LinearRegression().fit(mean_in[:, None], yy); res = yy - lin.predict(mean_in[:, None])
    dis = inner.std(0); sm = LinearRegression().fit(dis[:, None], np.abs(res)); floor = 0.5 * np.abs(res).mean()
    sigma = lambda d: np.maximum(sm.predict(d[:, None]), floor)
    fitted = [f(idx, yy, seed) for f in V6_COST]
    def pred(j, t=False):
        P3 = np.stack([g(j, t) for g in fitted])
        return lin.predict(P3.mean(0)[:, None]), sigma(P3.std(0))
    return pred, res / sigma(dis)

def decide_spread(p, mu, sig, z):
    """same decision rule as decide(), with cost samples exp(mu_i + sigma_i * z)"""
    z = np.sort(z); R = len(z); F, B = np.zeros(len(p), int), np.zeros(len(p))
    for s in range(0, len(p), 4000):
        e = slice(s, s + 4000); pe = p[e]; C = np.clip(np.exp(mu[e][:, None] + sig[e][:, None] * z[None, :]), 5, 8000)
        level = np.where(pe > 1 - Q_STAR, (pe - (1 - Q_STAR)) / np.maximum(pe, 1e-12), 0.0)
        idx = np.clip(np.ceil(level * R).astype(int) - 1, 0, R - 1)
        b = np.where(level > 0, C[np.arange(len(pe)), idx], 0.0)
        ec = INSPECT + pe * (RUSH * np.maximum(C - b[:, None], 0).mean(1) + RETURN * np.maximum(b[:, None] - C, 0).mean(1)) + (1 - pe) * RETURN * b
        f = (ec < pe * C.mean(1)).astype(int); F[e] = f; B[e] = np.where(f == 1, np.round(b, 2), 0.0)
    return F, B

### V6 versus V5 validation

Compare six alternative splits (seed 2025) using the same scenario assumptions, EM-derived odds and budget scaling. The underlying training observations are reused.

In [ ]:
final = make_folds(2025, 6); v5res = np.zeros((6, len(WORLDS))); v6res = v5res.copy(); r2 = np.zeros((6, 2)); failed = y == 1
for r, fs in enumerate(final):
    F5, B5, F6, B6 = np.zeros(len(y), int), np.zeros(len(y)), np.zeros(len(y), int), np.zeros(len(y))
    mu5, mu6 = np.zeros(len(y)), np.zeros(len(y))
    for k, (a, b) in enumerate(fs):
        idx = a[y[a] == 1]; seed = SEED + 10 * r + k; m = b[FULL[b]]
        o = ODDS * Pc[r % 3][m] / (1 - Pc[r % 3][m]); p = o / (1 + o)
        pred5, rs = fit_cost(V4_COST, idx, seed); mu5[b] = pred5(b); F5[m], B5[m] = decide(p, mu5[m], rs)
        pred6, z = fit_cost_spread(idx, seed); mu_b, sig_b = pred6(b); mu6[b] = mu_b; pos = np.searchsorted(b, m)
        F6[m], B6[m] = decide_spread(p, mu_b[pos], sig_b[pos], z)
    B5, B6 = np.minimum(B5 * 1.15, 8000), np.minimum(B6 * 1.15, 8000)
    for j, (pi, cs) in enumerate(WORLDS):
        v5res[r, j] = 20000 * (weights(pi) * fleet_cost(F5, B5, y, cost * cs)).sum()
        v6res[r, j] = 20000 * (weights(pi) * fleet_cost(F6, B6, y, cost * cs)).sum()
    for n, mu in enumerate([mu5, mu6]): r2[r, n] = 1 - ((mu[failed] - LOGC[failed]) ** 2).mean() / LOGC[failed].var()
    print("split", r, f"log-cost R2  V5 {r2[r,0]:.3f}  V6 {r2[r,1]:.3f}", f"{time.time()-T0:.0f}s")
idx = pd.MultiIndex.from_tuples(WORLDS, names=["failure rate", "cost level"])
display(pd.DataFrame({"V5 cost": v5res.mean(0), "V6 cost": v6res.mean(0), "V6 - V5": (v6res - v5res).mean(0),
                      "splits V6 wins (of 6)": (v6res < v5res).sum(0)}, index=idx).round(0))

### Final V6 submission generation

Average five fitted cost-model predictions and apply the calibrated failure probabilities and budget decision rule. Write `sub_v6.csv`.

In [ ]:
mus, sigs, zs = [], [], []
for s in range(5):
    pred, z = fit_cost_spread(fails, SEED + s)
    mu_s, sig_s = pred(np.arange(len(te)), True); mus.append(mu_s); sigs.append(sig_s); zs.append(z)
o = ODDS * pt_cal / (1 - pt_cal); p = o / (1 + o)
F, B = decide_spread(p, np.mean(mus, 0), np.mean(sigs, 0), np.concatenate(zs))
B = np.where(F == 1, np.minimum(B * 1.15, 8000), 0.0).round(2)
sub = pd.DataFrame({"id": te.id, "flag": F, "budget": B})
assert len(sub) == len(te) and (sub.id.values == te.id.values).all() and sub.flag.isin([0, 1]).all()
assert np.isfinite(sub.budget).all() and (sub.budget >= 0).all()
sub.to_csv(f"{OUT_DIR}/sub_v6.csv", index=False)
v5 = pd.read_csv(f"{OUT_DIR}/sub_v5.csv")
print(f"sub_v6.csv: flagged {F.sum()} (V5 {v5.flag.sum()}, in both {int(((F == 1) & (v5.flag == 1)).sum())}), budget sum {B.sum():,.0f} (V5 {v5.budget.sum():,.0f}), max {B.max():,.0f}")
print(f"total {time.time()-T0:.0f}s")

# Final Report: Fleet Maintenance Planning (ML Project 2)

**Team:** Machine Burning
**Competition:** Kaggle *ML Project 2*, Fleet Maintenance Planning
**Final submission:** `sub_v6.csv`, with a public score of **15,928.88**
**Our three best models:** **V5** (higher budgets), **V6** (final model) and **V7** (six classifiers)

From our earlier team pipeline to V6, the public cost came down by about **21%** (20,277 → 15,928.88).

---

## 1. The Problem

For each of the 20,000 test vehicles we have to decide two things, i.e. whether to inspect it (`flag`) and how much to set aside for spare parts (`budget`). The score is the total cost of these decisions, so a lower score is better.

| Situation | Cost |
|---|---|
| A vehicle is inspected | 10 |
| A failing vehicle is not inspected | the full `repair_cost` |
| The budget is too low | 0.20 for every missing unit |
| The budget is too high | 0.02 for every unused unit |

**About the data**

- **Training:** 56,000 vehicles from one depot, with `followup_days`, `failure` and `repair_cost`.
- **Test:** 20,000 vehicles from a **different depot**, with only the sensor readings.
- **Features:** 170 unnamed sensors with some missing values. We found seven groups of 10 sensors which behave like histograms.
- **Imbalance:** only **397** training vehicles failed, which is about 0.7%.
- **Follow-up:** a vehicle which looks healthy after a short follow-up may have failed later.

**How we evaluated our models:** we used cross-validation along with our own implementation of the competition cost, since plain accuracy tells us very little when failures are so rare. From V4 onwards we also tested on a simulated version of the new depot. The public leaderboard covers 30% of the test vehicles and the final ranking uses the other 70%.

---

## 2. Our Progress

| Version | Main idea | Test inspections | Local cost (per 20,000) | Public score |
|---|---|---:|---:|---:|
| Earlier pipeline | Ensemble with a simple cost estimate and EM | - | 16,662 (A) | 20,277 |
| V1 | Logistic regression used as a plain classifier | - | 52,998 (A) | not recorded |
| V2 | Calibrated LightGBM and a cost-based decision rule | 1,352 | 19,906 (A) | not recorded |
| V3a | Cost model for each vehicle and three classifiers | 1,940 | 14,256 (A) | 19,515 |
| V3b | Lower assumed failure rate and a new cost mix | - | - | 17,832 |
| V3c | A variation of V3b | - | - | slightly worse than V3b |
| V4a | All vehicles as reference and three cost models | 1,560 | 54,018 (B) | 17,407.88 |
| V4b | V4a with a lower odds multiplier | 1,444 | 54,546 (B) | 17,979.60 |
| **V5** | V4a with budgets × 1.15 | 1,560 | 55,163 (C) | **17,251.63** |
| **V6** | Seven histogram groups and a separate cost range per vehicle | 1,570 | 50,334 (C) | **15,928.88** |
| **V7** | V6 with six classifiers | 1,595 | - | **16,184.67** |

The local costs come from three different set-ups, so only values with the same letter can be compared:

- **(A)** cross-validation on the old depot;
- **(B)** a simulated new depot with 4% failures, on one set of data splits;
- **(C)** the same simulated depot on a different set of splits.

**Our journey in brief**

1. **Taking better decisions (V1 to V2):** choosing flags and budgets using the cost function.
2. **Understanding the new depot (V3 to V4):** correcting the sensor differences and the higher failure rate.
3. **Predicting repair costs better (V5 to V6):** adjusting the budgets and building much better cost features.

---

## 3. How We Got Here (V1 to V4)

### V1 and V2: the decisions mattered more than the model

V1 flagged a vehicle only when its failure probability was above 0.5 and gave every flagged vehicle the same budget. Its ranking of vehicles was already good (AUC 0.986), but it missed most of the failures. We then tried changing one thing at a time:

| Configuration | Local cost |
|---|---:|
| V1: logistic regression, 0.5 threshold, one budget | 52,998 |
| LightGBM with the same rule | 53,655 |
| Logistic regression with calibration and the cost-based rule | 22,749 |
| **V2: LightGBM with calibration and the cost-based rule** | **19,906** |

This clearly showed that most of the gain came from the decision rule and not from a stronger classifier.

### V3: a separate cost for each vehicle

V2 still assumed the same repair cost for everyone. We added the following one by one:

| Step | Local cost |
|---|---:|
| V2 | 19,906 |
| + a cost model for each vehicle | 15,685 |
| + zero weight for healthy vehicles with a short follow-up | 15,121 |
| + dropping the shifted sensors `s120` and `s122` | 14,612 |
| + LightGBM, XGBoost and CatBoost on stable sensors | **14,256** |

We also corrected for the higher failure rate of the new depot. In V3a we assumed about 4.8%, and in V3b we reduced it to 3.5% and changed the cost mix, which brought the public score from 19,515 to 17,832.

### V4: choosing the right reference

We found that vehicles with a short follow-up read much higher on the usage-like sensors and that the test depot looks like all the training vehicles together. Using all of them as the reference, the EM estimate gave an odds multiplier of 3.97 instead of about 5.4. V4a also combined three cost models. It scored 17,407.88, while V4b with a multiplier of 3.0 scored 17,979.60.

---

## 4. Selected Model 1: V5, Higher Budgets

**What we noticed in V4a:** a missing unit costs ten times more than an unused one, so a budget which is slightly too high is safer than one which is too low.

**Change:** we kept the inspections of V4a exactly the same and multiplied every budget by **1.15**, with a maximum of 8000.

**Results** (change in cost per 20,000 vehicles compared with V4a, on two sets of splits; negative is better):

| Assumed depot | Set 1 | Set 2 |
|---|---:|---:|
| Weighted over several assumed depots | −634 | −785 |
| Central assumption | −348 | −554 |
| Cheaper repairs and fewer failures | +377 | +398 |

**Observations**

- The total budget went up from about 875,000 to 1,001,000.
- The public score improved from 17,407.88 to **17,251.63**, which is a small gain of 156.
- It can make the cost worse if repairs are cheaper at the new depot.
- The weights of the assumed depots were estimated using our public scores, so this choice depends partly on the leaderboard.

**What was still left:** the budgets could only be as good as the repair cost predictions behind them.

---

## 5. Selected Model 2: V6, Better Repair Cost Predictions (Final Model)

**What was lacking in V5:** only one of the histogram-like sensor groups was being used as a histogram, and every vehicle shared the same range of cost errors.

### What V6 does

1. **Keeps the failure classifier:** the average of LightGBM, XGBoost and CatBoost, calibrated with Platt scaling, followed by the EM correction.
2. **Builds 163 histogram features** from the groups `s023–s032`, `s037–s046`, `s054–s063`, `s080–s089`, `s109–s118`, `s129–s138` and `s154–s163`.
3. **Averages three cost models:** RBF kernel ridge on 60 histogram features; Bayesian ridge with LightGBM learning its errors; and LightGBM on sensors, histogram features and 100 sensor ratios.
4. **Gives each vehicle its own range:** the range is wider when the three models disagree more.
5. **Decides using the cost function:** the budget is the 0.20/0.22 quantile of the predicted cost, allowing for the chance of no failure. A vehicle is inspected only if this lowers its expected cost. Finally the budgets are multiplied by 1.15.

### Results

**Simulated new depot** (cost per 20,000 vehicles, six splits not used for choosing the model):

| Failure rate | Price level | V5 | V6 | Difference |
|---|---|---:|---:|---:|
| 2.5% | 1.0 | 39,788 | **36,907** | −2,881 |
| 3.3% | 1.0 | 47,988 | **44,068** | −3,920 |
| 4.0% | 1.0 | 55,163 | **50,334** | −4,829 |
| 5.0% | 1.0 | 65,413 | **59,286** | −6,127 |
| 3.3% | 0.8 | 41,950 | **39,325** | −2,625 |
| 2.5% | 0.6 | 31,698 | **30,813** | −885 |
| 4.0% | 1.3 | 69,412 | **63,679** | −5,733 |

**Where the saving came from**

| Part of the cost | Change |
|---|---:|
| Budget too low | −4,181 |
| Missed failures | −1,435 |
| Budget too high | −885 |
| Inspection fees | +69 |

**Observations**

- V6 won on all six splits in every case above, and also on all eight splits of a further check.
- The R² on log repair cost went up from 0.775–0.791 to **0.825–0.835**.
- With the same linear model, the new features alone raised R² from about 0.72 to 0.79.
- Around three quarters of the saving came from the cost model and one quarter from the separate ranges.
- 1,570 vehicles are flagged, out of which 1,496 were also flagged by V5.
- The public score improved from 17,251.63 to **15,928.88**, i.e. by 7.7%.

---

## 6. Selected Model 3: V7, Six Classifiers

**Idea:** the classifier had not changed since V3, so we wanted to see whether adding shallower models would make it more reliable.

**Change:** we added three shallower versions of LightGBM, XGBoost and CatBoost and took the average of all six. The cost model is the same as V6.

**Results**

| Measure | Repeat 1 | Repeat 2 | Repeat 3 |
|---|---:|---:|---:|
| Average precision, V6 | 0.7710 | 0.7665 | 0.7788 |
| Average precision, V7 | 0.7745 | 0.7713 | 0.7803 |
| Simulated cost change from V6 | −719 | −12 | −1,030 |
| Share of vehicle resamples where V7 was better | 77% | 49% | 82% |

**Observations**

- V7 looked slightly better locally but did not reach the **90%** pass mark which we had set beforehand.
- Its public score was **16,184.67**, which is 256 worse than V6. This difference is small enough to be due to chance.
- We therefore kept V6 as our final model.

---

## 7. Other Ideas That Did Not Work

| What we tried | What happened |
|---|---|
| Increasing budgets beyond 1.15 | Our results did not support it |
| A cost-weighted mix of classifiers | It improved only one of three repeats |
| Stacking | It did not meet our pass mark |
| More cost models and bagging | R² did not improve |
| Histogram features in the classifier | No improvement |

---

## 8. How the Data Is Used

**Fitted on `train.csv` only:** all the V6 failure and repair cost models, along with the scaling, median filling and feature selection used by them.

**Calculated row by row for both files:** the histogram features.

**How `test.csv` is used**

- **Predictions:** the trained models are applied to the test vehicles.

- **Diagnostic:** our first analysis (`fleet_final_report.ipynb`) trained a classifier to tell train rows from test rows. No test labels were available.





---

## 9. Our Best Model (V6): Strengths and Limitations

**Strengths**

- The decisions follow directly from the competition costs.
- It predicts repair costs much better than our earlier versions.
- It accounts for being more unsure about some vehicles than others.
- It uses models of different types, which make different kinds of errors.

**Limitations**

- With only 397 failures our estimates are noisy, and much of the gain comes from a few expensive repairs.
- The same failures were used repeatedly during development, so offline results may be optimistic.
- Our simulations expected larger gains than we got, e.g. about 1,600 (median) for V6 against the actual 1,323.
- The repair prices of the new depot cannot be checked.
- The public score covers only 30% of the test vehicles, so the private score may differ.

---

## 10. Individual Contributions

| Member | Work |
|RIGZIN GYALPO| V4 notebook|
| HARSHUL KUMAR MAURYA| V2 and V7 notebook|
| DHRUV PESHIN |  V1 AND V3 Combined Notebook |
| HANSIKA SINGH | report and v6 notebook |

---

## 11. Repository and How to Run

```
fleet_final_report.ipynb      V1, V2 and V3a
notebooks/fleet_v4.ipynb      V4a and V4b
notebooks/fleet_v5.ipynb      V5
notebooks/fleet_v6.ipynb      final model V6
notebooks/fleet_v6_executed.ipynb   original saved run of V6
notebooks/fleet_v6_general.ipynb    V6 with automatic depot checks
notebooks/fleet_v7.ipynb      V7, which also creates V5 and V6
best_submissions/             sub_v5.csv, sub_v6.csv, sub_v7.csv
SUBMISSIONS.md                submission log
experiments/                  earlier exploratory scripts
```


